# **Cómo funciona optuna**

Funciona con 3 conceptos clave, un **study**, que viene a ser el experimento completo, **trials**, cada evaluación de hiperparámetros, y una **función objetivo** (la función que se intenta minimizar/maximizar).

In [1]:
%pip install optuna
%pip install optuna-dashboard plotly

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [14]:
import torch
import torch.nn as nn # Modulo para redes neuronales
import torch.optim as optim # Modulo para optimizadores
import optuna # Modulo para optimización de hiperparámetros
from optuna.pruners import MedianPruner

# Modulo para cargar y transformar datos
import torchvision
from torchvision import datasets, transforms, models

# Modulo para visualización
import matplotlib.pyplot as plt
import numpy as np

# Modulo para evaluación de modelos
from sklearn.metrics import classification_report, confusion_matrix

import os

In [11]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

# **Carga de los datos**

In [4]:
raiz = '../../data/Ojos'

# Datos de entrenamiento
train_dir = os.path.join(raiz, 'train')
train_dir_normal = os.path.join(train_dir, 'Normales')
train_dir_glaucoma = os.path.join(train_dir, 'Glaucomas')

# Datos de validación
val_dir = os.path.join(raiz, 'val')
val_dir_normal = os.path.join(val_dir, 'Normales')
val_dir_glaucoma = os.path.join(val_dir, 'Glaucomas')

# Datos de prueba
test_dir = os.path.join(raiz, 'test')
test_dir_normal = os.path.join(test_dir, 'Normales')
test_dir_glaucoma = os.path.join(test_dir, 'Glaucomas')

# Muestro la cantidad de imágenes en cada conjunto
suma_entrenamiento = len(os.listdir(train_dir_normal)) + len(os.listdir(train_dir_glaucoma))
suma_validacion = len(os.listdir(val_dir_normal)) + len(os.listdir(val_dir_glaucoma))
suma_prueba = len(os.listdir(test_dir_normal)) + len(os.listdir(test_dir_glaucoma))

print("Datos de entrenamiento:", suma_entrenamiento, "--- Normales:", len(os.listdir(train_dir_normal)), "--- Glaucoma:", len(os.listdir(train_dir_glaucoma)))
print("Datos de validación:", suma_validacion, "--- Normales:", len(os.listdir(val_dir_normal)), "--- Glaucoma:", len(os.listdir(val_dir_glaucoma)))
print("Datos de prueba:", suma_prueba, "--- Normales:", len(os.listdir(test_dir_normal)), "--- Glaucoma:", len(os.listdir(test_dir_glaucoma)))

Datos de entrenamiento: 339 --- Normales: 219 --- Glaucoma: 120
Datos de validación: 48 --- Normales: 31 --- Glaucoma: 17
Datos de prueba: 98 --- Normales: 63 --- Glaucoma: 35


# **Data Augmentation**

In [7]:
# Data augmentation para el conjunto de entrenamiento
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(), # Volteo horizontal aleatorio
    transforms.RandomVerticalFlip(), # Volteo vertical aleatorio
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0), ratio=(1.0, 1.0)), # Zoom aleatorio
    transforms.RandomRotation(20), # Rotación aleatoria
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Para el conjunto de validacion
val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Para el conjunto de prueba
test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

In [9]:
train_dataset = datasets.ImageFolder(train_dir, transform=train_transforms)
val_dataset = datasets.ImageFolder(val_dir, transform=val_transforms)
test_dataset = datasets.ImageFolder(test_dir, transform=test_transforms)

In [10]:
train_loader = torch.utils.data.DataLoader(train_dataset,
                                           batch_size=16,
                                           shuffle=True)

val_loader = torch.utils.data.DataLoader(val_dataset,
                                         batch_size=16,
                                         shuffle=False)

test_loader = torch.utils.data.DataLoader(test_dataset,
                                          batch_size=1,
                                          shuffle=False)

# **Definición de la función objetivo**

In [12]:
def objective(trial):
    # ── Hiperparámetros a explorar ──────────────────────────────────────────
    lr          = trial.suggest_float("lr", 1e-5, 1e-3, log=True)
    batch_size  = trial.suggest_categorical("batch_size", [8, 16, 32])
    optimizer_name = trial.suggest_categorical("optimizer", ["Adam", "AdamW"])
    weight_decay   = trial.suggest_float("weight_decay", 1e-5, 1e-2, log=True)

    # ── DataLoaders con el batch_size del trial ─────────────────────────────
    train_loader_t = torch.utils.data.DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True
    )
    val_loader_t = torch.utils.data.DataLoader(
        val_dataset, batch_size=batch_size, shuffle=False
    )

    # ── Modelo: cargamos VGG19 fresco en cada trial ─────────────────────────
    model = models.vgg19(pretrained=True)
    model.classifier[6] = nn.Linear(4096, 2)

    # Fine-tuning: solo entrenamos el clasificador
    for param in model.features.parameters():
        param.requires_grad = False
    for param in model.classifier.parameters():
        param.requires_grad = True

    model.to(device)

    # ── Optimizador ─────────────────────────────────────────────────────────
    optimizer_cls = getattr(optim, optimizer_name)
    optimizer_t = optimizer_cls(
        model.parameters(), lr=lr, weight_decay=weight_decay
    )

    criterion = nn.CrossEntropyLoss()

    # ── Entrenamiento corto para buscar hiperparámetros (10 épocas) ─────────
    # No necesitas 100 épocas aquí, Optuna solo necesita una señal relativa
    N_EPOCHS_SEARCH = 10
    best_val_loss = float("inf")

    for epoch in range(N_EPOCHS_SEARCH):
        # Entrenamiento
        model.train()
        for images, labels in train_loader_t:
            images, labels = images.to(device), labels.to(device)
            optimizer_t.zero_grad()
            loss = criterion(model(images), labels)
            loss.backward()
            optimizer_t.step()

        # Validación
        model.eval()
        val_loss_epoch = 0.0
        with torch.no_grad():
            for images, labels in val_loader_t:
                images, labels = images.to(device), labels.to(device)
                val_loss_epoch += criterion(model(images), labels).item()

        val_loss_epoch /= len(val_loader_t)

        if val_loss_epoch < best_val_loss:
            best_val_loss = val_loss_epoch

        # Pruning: cancela trials poco prometedores temprano ✂️
        trial.report(val_loss_epoch, epoch)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return best_val_loss

Una vez definida la función objetivo la llamamos para que haga las pruebas.

In [15]:
study = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=MedianPruner(n_warmup_steps=3),  # no poda antes de la época 3
    # storage="sqlite:///glaucoma_optuna.db",  # descomenta para persistencia
    # study_name="vgg19_glaucoma",
    # load_if_exists=True,
)

study.optimize(objective, n_trials=20)  # 20 trials suele ser suficiente para estos params

print("\n── Mejores hiperparámetros ──────────────────")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")
print(f"  → Val loss: {study.best_value:.4f}")

[I 2026-04-28 18:43:04,841] A new study created in memory with name: no-name-b1147c44-2ea2-4fb8-a827-94a77ce59984
/home/fran/Redes-Neuronales-en-Keras/.venv/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/fran/Redes-Neuronales-en-Keras/.venv/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG19_Weights.IMAGENET1K_V1`. You can also use `weights=VGG19_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
[W 2026-04-28 18:47:36,329] Trial 0 failed with parameters: {'lr': 5.6115164153345e-05, 'batch_size': 8, 'optimizer': 'Adam', 'weight_decay': 1.493656855461762e-05} because of the following error: Keyboard

KeyboardInterrupt: 